# **PromptKaban: Semantic Search Engine**

This notebook builds and evaluates a **semantic search engine** over a corpus of **20,000 AI prompts**. 

Users can ask questions in plain language, and the engine finds the best matching prompts using:

- Embeddings from **nomic-embed-text-v1.5**
- A spectral graph index from **ArrowSpace**
- An engagement-aware reranker (MMR + salience from upvotes, likes, reputation, views)

We start by looking at the raw data to understand what the model sees.

## 1. Data preview

The dataset is stored as JSON:

- `dataset.json`: 20,000 prompts with content and metadata  
- `queries.json`: evaluation queries, each with an `expected_prompt_id`

Below we sample one random prompt and one random query to check the structure.

In [6]:
!jq ".[$(jq 'keys[]' ../data/dataset.json | shuf -n 1)]" ../data/dataset.json

{
  "id": "pk_16762",
  "author_reputation": 17,
  "version": 2,
  "fork_count": 2,
  "likes": 17,
  "upvotes": 11,
  "downvotes": 1,
  "views": 173,
  "uses": 17,
  "created_at": "2024-09-09T01:58:39Z",
  "title": "FIND MY RESEARCH GAP",
  "content": "im doing a lit review on AI IN HEALTHCARE tell me what angles are missing so I can fill the research gap",
  "category": "science-research",
  "subcategory": "literature-review",
  "tags": [
    "ai",
    "healthcare",
    "research-gap",
    "systematic-review"
  ],
  "has_placeholders": false,
  "placeholders": [],
  "difficulty": "advanced",
  "language": "en",
  "target_model": "llama-3.3-70b"
}


In [7]:
!jq ".[$(jq 'keys[]' ../data/queries.json | shuf -n 1)]" ../data/queries.json

{
  "query_id": "q_02",
  "query_text": "Our customer satisfaction survey scores went down this quarter and I need help analyzing the text responses to fix it.",
  "expected_prompt_id": "pk_19098",
  "query_type": "Semantic & Intent-Based"
}


The sample prompt `pk_16762` shows the usual fields we will use: 
- Text fields for meaning: `title`, `content`, `category`, `subcategory`, `tags` 
- Signals for quality: `upvotes`, `likes`, `author_reputation`, `views` 

The sample query (like `q_02`) shows how we check things. For each `query_text`, we know which prompt ID (`expected_prompt_id`) to find.

## 2. Structured document representation

If we only use the raw `content` field, many prompts are too short or unclear to match real queries well. 

To fix this, we build a single **`doc_string`** for each prompt that includes all useful fields:

- Title
- Category and subcategory
- Difficulty
- Tags
- Full content

In [1]:
import json 
from pathlib import Path 
from tinydb import TinyDB, Query
import re
import numpy as np
import arrowspace_tuner


db = TinyDB('db.json')
ROOT = Path.cwd().parent 
dataset_path = ROOT / "data" / "dataset.json"
query_path = ROOT / "data" / "nomic_embs" / "queries_emb_768.npy"
query_index_path = ROOT / "data" / "nomic_embs" / "queries_index.json"

DOC_PREFIX = "search_document: "

/home/tommaso/code_base/prompt_kaban_engine/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open(dataset_path, "r") as f:
    dataset = json.load(f)

In [3]:
prompts = Query()

In [4]:
def clean(text: str) -> str:
    """Remove {{placeholder}} tokens, collapse whitespace."""
    return re.sub(r"\s+", " ", re.sub(r"\{\{[^}]+\}\}", "", text)).strip()


def is_weak_title(el: dict) -> bool:
    """True when title has ≤4 words or shares no token with its own tags/category."""
    t = el["title"].lower()
    signals = el.get("tags", []) + [el.get("category", ""), el.get("subcategory", "")]
    has_signal = any(s.lower().replace("-", " ") in t for s in signals if s)
    return len(el["title"].split()) <= 4 or not has_signal


def extract_structured(el: dict) -> str:
    """
    Richest form: explicit field labels + full content.
    Named fields (Title:, Category:, Tags:) guide nomic attention to distribute
    semantic load evenly across ALL Matryoshka bands.
    Primary fix for Type-A failures (branding, coaching, communication, audit):
    engine is completely blind — needs every contextual signal available.
    """
    title  = el["title"].strip()
    tags   = ", ".join(el["tags"])
    body   = clean(el["content"])
    cat    = el["category"]
    subcat = el.get("subcategory", "")
    diff   = el.get("difficulty", "general")
    ph     = el.get("placeholders", [])
    ph_str = f"Variables: {', '.join(ph)}.\n" if ph else ""

    return (
        DOC_PREFIX
        + f"Title: {title}\n"
          f"Category: {cat} > {subcat}\n"
          f"Difficulty: {diff}\n"
          f"Tags: {tags}\n"
        + ph_str
        + body
    ).strip()


In [6]:
for el in dataset:
    prompt = extract_structured(el)
    db.insert({'id' : el["id"], 'doc_string': prompt})

In [17]:
db.search(prompts.id == "pk_03138")

[{'id': 'pk_03138',
  'doc_string': 'search_document: Title: Complete Brand Identity Brief\nCategory: brand-identity > strategic-brief\nDifficulty: expert\nTags: brand-brief, sustainable-fashion, strategy, identity\nCreate a comprehensive brand identity brief document for a new sustainable fashion startup. Include all essential sections: Executive Summary, Brand Purpose and Mission, Target Audience Persona (demographics, psychographics, behaviors), Brand Positioning statement, Brand Personality (with detailed trait descriptions), Competitive Analysis summary, Core Brand Values, Brand Promise, Tone of Voice guidelines (with examples for different scenarios), and Visual Direction overview. Format this as a professional document a design agency would deliver to stakeholders.'},
 {'id': 'pk_03138',
  'doc_string': 'search_document: Title: Complete Brand Identity Brief\nCategory: brand-identity > strategic-brief\nDifficulty: expert\nTags: brand-brief, sustainable-fashion, strategy, identity

This format gives the embedding model a lot more signal than the raw content alone. Every prompt is stored in TinyDB as `{id, doc_string}`, ready for the embedding step.

## **3. Load embeddings and build ArrowSpace**

We now move from **raw text** to **vector space** and build the spectral index.

### **3.1 Data pipeline**

The main work (encoding and saving embeddings) is done once:

**Offline steps:**

1. We store all prompts in TinyDB (`db.json`) as `{"id": ..., "doc_string": ...}` for 20k records.

2. `scripts/nomic_corpus_embs.py` reads those records, runs `nomic-embed-text-v1.5` on each `doc_string`, and saves:
    - `embeddings_nomic_structured_768d_raw.npy`  → `(N, 768)` float32
    - `embeddings_nomic_structured_768d_ids.npy`  → `(N,)` str, same row order
    - `embeddings_registry.json` → tells us, for each ID, which row and which file to load

3. In this notebook we load those `.npy` files and call `ArrowSpaceBuilder.build(...)` to create:
    - `aspace` → the ArrowSpace index (with one λ value per prompt)
    - `gl` → the Graph Laplacian reused at query time

We always keep IDs and embeddings in the **same row order**, so `ids[i]` matches `embs[i]` and `aspace.lambdas()[i]`. 
That alignment is checked with an assert.


**Memory note (float64 in RAM)**
| Prompts | 256d  | 512d  | 768d   |
|---------|-------|-------|--------|
| 20k     | 40 MB | 80 MB | 120 MB |
| 100k    | 200 MB| 400 MB| 600 MB |
| 500k    | 1 GB  | 2 GB  | 3 GB   |
| 1M      | 2 GB  | 4 GB  | 6 GB   |

ArrowSpace keeps an internal copy of the matrix, so you can roughly **double** these numbers. 

For 1M prompts, 256d fits on a 16 GB machine; 768d is more comfortable on 32 GB. The sparse Laplacian adds only ~50 MB on top.

In [8]:
import numpy as np
from pathlib import Path

EMBS_DIR = ROOT / "data" / "nomic_embs" 
DIM      = 768  

emb_path = EMBS_DIR / f"embeddings_nomic_structured_{DIM}d_raw.npy"
ids_path = EMBS_DIR / f"embeddings_nomic_structured_{DIM}d_ids.npy"

# ArrowSpace requires float64
embs = np.load(emb_path).astype(np.float64)
ids  = list(np.load(ids_path))

# alignment guard — must never fail
assert embs.shape[0] == len(ids), \
    f"Shape mismatch: embs {embs.shape[0]} rows vs {len(ids)} ids"

print(f"Embeddings : {embs.shape}  dtype={embs.dtype}")
print(f"IDs        : {len(ids)} entries")
print(f"RAM in use : {embs.nbytes / 1e9:.2f} GB")
print(f"Sample     : ids[0]={ids[0]!r}  embs[0,:4]={embs[0, :4]}")

Embeddings : (20000, 768)  dtype=float64
IDs        : 20000 entries
RAM in use : 0.12 GB
Sample     : ids[0]=np.str_('pk_00001')  embs[0,:4]=[ 0.76463675  0.68402106 -3.29455972 -0.1249674 ]


The corpus embeddings load as a `(20_000, 768)` matrix in `float64`, using about **0.12 GB**.  
The check `len(ids) == embs.shape[0]` makes sure that every row in the matrix has a matching prompt ID. This is necessary before we build any index.

### **3.2 ArrowSpace hyperparameter tuning**

ArrowSpace builds a k‑NN graph and a graph Laplacian over the embedding space. It gives each prompt a **Rayleigh quotient λ** that shows how central or peripheral it is in the semantic manifold.

To avoid hand-tuning graph parameters, we use `EpsTuner` to search over:

- `eps` ∈ [0.5, 2.0]: neighbourhood radius  
- `k`: effective number of nearest neighbours  
- `topk`: candidate list size  
- `p`: distance norm

It runs 15 tests on a 5,000‑sample and stores results in a local SQLite DB for reproducibility (`arrowspace_tuning.db`).

In [9]:
from arrowspace_tuner import EpsTuner

tuner = EpsTuner(
    n_trials  = 15,
    sample_n  = 5_000,
    eps_low   = 0.5,
    eps_high  = 2.0,
    tau_low   = 0.42,
    tau_high  = 1.0,
    n_probe   = 50,
    storage   = "sqlite:///arrowspace_tuning.db",
)

In [ ]:
aspace, gl = tuner.fit(embs)
tuner.save_report()

In [10]:
graph_params = tuner.load_best_params()

graph_params

{'eps': 1.2310568997881801, 'k': 38, 'topk': 19, 'p': 2.0, 'sigma': None}

The tuner found these values:

```python
{'eps': 1.2311, 'k': 38, 'topk': 19, 'p': 2.0, 'sigma': None}
```

This means we have a graph with about 38 neighbors for each point. This is good for a set of 20,000 prompts that includes both popular topics and less common ones. We now pass these parameters to `ArrowSpaceBuilder` to build the full index.

In [ ]:
from arrowspace import ArrowSpaceBuilder
aspace, gl = ArrowSpaceBuilder().build(graph_params, embs)

`aspace` now holds the ArrowSpace index, and `gl` holds the corresponding graph Laplacian.  

Every prompt has:
- an embedding vector in `embs[i]`
- a prompt ID in `ids[i]`
- a spectral energy score λᵢ available via `aspace.lambdas()[i]`

In the next section we inspect the graph structure to understand how the dataset is wired in the spectral domain before running any retrieval experiments.

## Arrowspace Dataset monitoring

embeddings energy stats

In [13]:
# aspace_base stats
lambdas = aspace.lambdas()
print("aspace_base (semantic only)")
print(f"  λ  min={min(lambdas):.6f}  max={max(lambdas):.6f}  "
      f"mean={sum(lambdas)/len(lambdas):.6f}")

aspace_base (semantic only)
  λ  min=0.000000  max=1.000000  mean=0.124252


Graph Laplacian

In [14]:
import numpy as np
import plotly.graph_objects as go
import scipy.sparse as sp
from scipy.sparse.linalg import eigsh
from sklearn.decomposition import PCA
from scipy.interpolate import griddata
from scipy.ndimage import gaussian_filter

# ── point to the meta-augmented embeddings & Laplacian ────────────────
rows = embs 

# ── Helpers ────────────────────────────────────────────────────────────
def gl_to_scipy(gl) -> sp.csr_matrix:
    raw = gl.to_csr()
    return sp.csr_matrix(
        (np.asarray(raw[0]), np.asarray(raw[1]), np.asarray(raw[2])),
        shape=gl.shape()
    )

L       = gl_to_scipy(gl)   # gl = gl_meta alias set in the build cell
n_nodes = L.shape[0]
degrees = np.array(L.diagonal(), dtype=np.float64)




# ── Clip outliers ──────────────────────────────────────────────────────
p95  = np.percentile(degrees, 95)
p05  = np.percentile(degrees, 5)
z_pts = np.clip(degrees, p05, p95)


# ── 2D layout: rows (PCA) if available, else spectral embedding ────────
try:
    # 'rows' must be a (N, D) array-like already in scope
    if 'rows' not in dir() or rows is None:
        raise NameError("rows not defined")
    embs_2d  = np.array(rows[:n_nodes], dtype=np.float32)
    pca      = PCA(n_components=2)
    xy       = pca.fit_transform(embs_2d)
    x_pts, y_pts = xy[:, 0], xy[:, 1]
    var          = pca.explained_variance_ratio_
    x_label      = f'PC1 ({var[0]*100:.1f}%)'
    y_label      = f'PC2 ({var[1]*100:.1f}%)'
    subtitle     = f'PC1 {var[0]*100:.1f}%  PC2 {var[1]*100:.1f}%'

except NameError:
    # ── Fallback: Fiedler spectral embedding from L itself ────────────
    # Use the 2nd & 3rd smallest eigenvectors (skip trivial constant vec)
    k_eig = min(4, n_nodes - 1)          # need at least 3 eigenvectors
    vals, vecs = eigsh(
        L.astype(np.float64), k=k_eig, which='SM', tol=1e-5, maxiter=3000
    )
    order     = np.argsort(vals)         # sort ascending by eigenvalue
    vecs      = vecs[:, order]
    x_pts     = vecs[:, 1]              # Fiedler vector  (λ₂)
    y_pts     = vecs[:, 2]              # next eigenvector (λ₃)
    var       = None
    x_label   = f'Fiedler v (λ₂={vals[order[1]]:.3f})'
    y_label   = f'Spectral v (λ₃={vals[order[2]]:.3f})'
    subtitle  = 'spectral embedding  ·  Fiedler (λ₂) × λ₃'


# ── Interpolation on a fine grid ───────────────────────────────────────
grid_res = 120
xi = np.linspace(x_pts.min(), x_pts.max(), grid_res)
yi = np.linspace(y_pts.min(), y_pts.max(), grid_res)
Xi, Yi = np.meshgrid(xi, yi)

Zi      = griddata((x_pts, y_pts), z_pts, (Xi, Yi), method='cubic')
Zi_near = griddata((x_pts, y_pts), z_pts, (Xi, Yi), method='nearest')
Zi[np.isnan(Zi)] = Zi_near[np.isnan(Zi)]
Zi = gaussian_filter(Zi, sigma=2.5)


# ── Colorscale ────────────────────────────────────────────────────────
colorscale = [
    [0.0,  '#1a3a6b'],
    [0.25, '#4a90d9'],
    [0.5,  '#f5f5f5'],
    [0.75, '#e05c3a'],
    [1.0,  '#8b0000'],
]


# ── Surface ───────────────────────────────────────────────────────────
surface = go.Surface(
    x=Xi, y=Yi, z=Zi,
    colorscale=colorscale,
    opacity=1.0,
    showscale=True,
    colorbar=dict(
        title=dict(text='Degree (Curvature)', font=dict(size=12)),
        thickness=16, x=1.01, tickfont=dict(size=10),
    ),
    contours=dict(
        z=dict(
            show=True,
            usecolormap=True,
            highlightcolor='rgba(255,255,255,0.6)',
            project_z=True,
            start=float(Zi.min()),
            end=float(Zi.max()),
            size=(Zi.max() - Zi.min()) / 12,
        )
    ),
    lighting=dict(ambient=0.7, diffuse=0.9, specular=0.2,
                  roughness=0.6, fresnel=0.1),
    lightposition=dict(x=1000, y=1000, z=2000),
    hoverinfo='skip',
    name='Manifold',
)


# ── Hub-node scatter (top 15 % by degree) ─────────────────────────────
top_idx = np.where(degrees > np.percentile(degrees, 85))[0]
node_scatter = go.Scatter3d(
    x=x_pts[top_idx], y=y_pts[top_idx], z=z_pts[top_idx] + 1,
    mode='markers',
    marker=dict(
        size=5,
        color=degrees[top_idx],
        colorscale=colorscale,
        cmin=p05, cmax=p95,
        opacity=1.0,
        line=dict(width=0.8, color='white'),
    ),
    text=[f'Node {i}<br>degree={degrees[i]:.2f}' for i in top_idx],
    hoverinfo='text',
    name='Hub nodes (top 15%)',
)


# ── Figure ─────────────────────────────────────────────────────────────
fig = go.Figure(
    data=[surface, node_scatter],
    layout=go.Layout(
        title=dict(
            text=(
                f'Graph Laplacian Manifold  ·  {n_nodes} nodes<br>'
                f'<sup>Z = degree (local curvature, clip p5–p95)  ·  {subtitle}</sup>'
            ),
            x=0.5, font=dict(size=13, color='#e2e6f0'),
        ),
        scene=dict(
            xaxis=dict(title=x_label,
                       showgrid=True, gridcolor='rgba(100,100,160,0.25)',
                       backgroundcolor='rgb(15,15,30)', zeroline=False),
            yaxis=dict(title=y_label,
                       showgrid=True, gridcolor='rgba(100,100,160,0.25)',
                       backgroundcolor='rgb(15,15,30)', zeroline=False),
            zaxis=dict(title='Degree L_ii',
                       showgrid=True, gridcolor='rgba(100,100,160,0.25)',
                       backgroundcolor='rgb(15,15,30)', zeroline=False),
            bgcolor='rgb(10,10,20)',
            camera=dict(eye=dict(x=1.6, y=1.6, z=0.9),
                        up=dict(x=0, y=0, z=1)),
            aspectmode='manual',
            aspectratio=dict(x=1.6, y=1.6, z=0.55),
        ),
        paper_bgcolor='rgb(12,12,22)',
        font=dict(color='#e2e6f0', family='system-ui, sans-serif'),
        legend=dict(x=0.02, y=0.98, bgcolor='rgba(0,0,0,0.5)',
                    font=dict(size=11)),
        margin=dict(l=0, r=0, t=90, b=0),
        height=700,
    )
)

fig.show(renderer="vscode")

ADD a descriptions of the laplacian 

In [15]:

import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla

n          = L.shape[0]
nnz        = L.nnz
n_edges    = (nnz - n) // 2          # off-diagonal non-zeros / 2
sparsity   = 1.0 - nnz / (n * n)
avg_degree = float(degrees.mean())
std_degree = float(degrees.std())
max_degree = float(degrees.max())
min_degree = float(degrees.min())
degree_cv  = std_degree / avg_degree  # coefficient of variation: 0=flat, >1=spiky

# ── 2. Fiedler value (algebraic connectivity) ─────────────────────────────
# λ₁=0 always (constant vector); λ₂=Fiedler: how well-connected the graph is
try:
    diag_d     = np.array(L.diagonal(), dtype=np.float64)
    safe_d     = np.where(diag_d > 1e-12, diag_d, 1e-12)
    d_inv_sqrt = sp.diags(1.0 / np.sqrt(safe_d))
    L_norm     = d_inv_sqrt @ L @ d_inv_sqrt
    eigs       = spla.eigsh(L_norm, k=6, which="SM",
                            return_eigenvectors=False, tol=1e-5, maxiter=3000)
    eigs_sorted   = sorted(np.real(eigs))
    fiedler       = max(0.0, eigs_sorted[1])
    spectral_gap  = eigs_sorted[2] - eigs_sorted[1] if len(eigs_sorted) > 2 else 0.0
except Exception as e:
    fiedler, spectral_gap = 0.0, 0.0
    print(f"[warn] Fiedler computation failed: {e}")

# ── 3. Degree distribution buckets ────────────────────────────────────────
p10, p25, p50, p75, p90 = np.percentile(degrees, [10, 25, 50, 75, 90])
hub_count      = int((degrees > p90).sum())            # top 10% = hubs
isolated_count = int((degrees < p10).sum())            # bottom 10% = tails
hub_fraction   = hub_count / n * 100
tail_fraction  = isolated_count / n * 100

# ── 4. Connectivity interpretation ────────────────────────────────────────
def interpret_fiedler(f):
    if f < 0.01:  return "⚠️  Near-disconnected (multiple isolated clusters)"
    if f < 0.05:  return "🟡 Weakly connected (sparse bridge structure)"
    if f < 0.15:  return "🟢 Moderately connected (balanced topology)"
    return               "🔵 Strongly connected (dense, cohesive manifold)"

def interpret_cv(cv):
    if cv < 0.3:  return "Flat (uniform density, no dominant hubs)"
    if cv < 0.7:  return "Moderate variation (some hubs, mostly uniform)"
    if cv < 1.2:  return "High variation (clear hub/tail structure)"
    return               "Power-law-like (few mega-hubs, many isolated tails)"

def interpret_gap(gap):
    if gap < 0.02: return "Continuous spectrum (smooth manifold)"
    if gap < 0.1:  return "Soft cluster boundary (overlapping topics)"
    return                "Hard cluster boundary (distinct topic groups)"

# ── 5. Prepare Table Data ─────────────────────────────────────────────────
rows_table = [
    ("Nodes (centroids)",        f"{n}",                              "Clusters extracted from your embeddings"),
    ("Edges (connections)",      f"{n_edges:,}",                      "Semantic proximity links between centroids"),
    ("Sparsity",                 f"{sparsity*100:.2f}%",              "Most centroids are NOT directly connected"),
    ("Avg degree",               f"{avg_degree:.3f}",                 "Mean connectivity per centroid"),
    ("Degree std",               f"{std_degree:.3f}",                 "Spread of connectivity across centroids"),
    ("Degree CV",                f"{degree_cv:.3f}",                  interpret_cv(degree_cv)),
    ("Max degree (hub)",         f"{max_degree:.3f} (node {int(np.argmax(degrees))})", "Most connected centroid"),
    ("Min degree (tail)",        f"{min_degree:.3f} (node {int(np.argmin(degrees))})", "Most isolated centroid"),
    ("Hub centroids (top 10%)",  f"{hub_count} ({hub_fraction:.1f}% of nodes)",       "Dominant theme anchors"),
    ("Tail centroids (bot 10%)", f"{isolated_count} ({tail_fraction:.1f}% of nodes)", "Sparse / hard-to-retrieve zones"),
    ("Fiedler value λ₂",         f"{fiedler:.5f}",                    interpret_fiedler(fiedler)),
    ("Spectral gap λ₃−λ₂",       f"{spectral_gap:.5f}",               interpret_gap(spectral_gap)),
] 

# ── 6. Print Plain Text Table ─────────────────────────────────────────────
print("\n" + "═" * 105)
print(" 📊 GRAPH LAPLACIAN — DATASET FINGERPRINT")
print("═" * 105)
print(f" {'METRIC':<25} │ {'VALUE':<20} │ {'INTERPRETATION'}")
print("─" * 105)
for row in rows_table:
    print(f" {row[0]:<25} │ {row[1]:<20} │ {row[2]}")
print("═" * 105)

# ── 7. One-paragraph narrative ────────────────────────────────────────────
print("\n── NARRATIVE SUMMARY ──────────────────────────────────────────────────────────")
print(f"""
Dataset contains {n} semantic centroids wired into a graph with {n_edges:,} edges 
({sparsity*100:.1f}% sparse).

Degree distribution: avg={avg_degree:.2f} ± {std_degree:.2f} (CV={degree_cv:.2f})
→ {interpret_cv(degree_cv)}

{hub_count} hub centroids ({hub_fraction:.1f}%) anchor the dominant themes.
{isolated_count} tail centroids ({tail_fraction:.1f}%) are the hard-to-retrieve zone —
exactly where ArrowSpace's spectral re-weighting adds value over flat cosine.

Algebraic connectivity (Fiedler λ₂ = {fiedler:.5f}):
→ {interpret_fiedler(fiedler)}

Spectral gap (λ₃ − λ₂ = {spectral_gap:.5f}):
→ {interpret_gap(spectral_gap)}
""")


═════════════════════════════════════════════════════════════════════════════════════════════════════════
 📊 GRAPH LAPLACIAN — DATASET FINGERPRINT
═════════════════════════════════════════════════════════════════════════════════════════════════════════
 METRIC                    │ VALUE                │ INTERPRETATION
─────────────────────────────────────────────────────────────────────────────────────────────────────────
 Nodes (centroids)         │ 768                  │ Clusters extracted from your embeddings
 Edges (connections)       │ 6,980                │ Semantic proximity links between centroids
 Sparsity                  │ 97.50%               │ Most centroids are NOT directly connected
 Avg degree                │ 10.343               │ Mean connectivity per centroid
 Degree std                │ 31.614               │ Spread of connectivity across centroids
 Degree CV                 │ 3.057                │ Power-law-like (few mega-hubs, many isolated tails)
 Max degree (

### Lambdas distribution

In [20]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Using your stats from the image_f3e3db.png terminal output
# mean: 0.128270, median: 0.045436, p95: ~0.45

# --- 1. Clipping Strategy ---
x_limit = np.percentile(lambdas, 60)  # Focus on the 95% bulk
lambdas_clipped = lambdas[lambdas <= x_limit]
tail_count = len(lambdas) - len(lambdas_clipped)
tail_pct = (tail_count / len(lambdas)) * 100

# Higher resolution for the bulk of the data
bins = np.linspace(0, x_limit, 200)
h, edges = np.histogram(lambdas, bins=bins)
centers = (edges[:-1] + edges[1:]) / 2

# --- 2. Figure Construction ---
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=(
        f"λ Density — Bulk Distribution (Clipped at p60)",
        "ECDF — Linear Scale with Progress Zones"
    ),
    vertical_spacing=0.15,
)

# Row 1: Histogram (Linear but Focused)
fig.add_trace(go.Bar(
    x=centers, y=h,
    name='Base Distribution',
    marker=dict(
        color=centers,
        colorscale='Viridis',
        showscale=False,
        line_width=0
    ),
    hovertemplate='λ: %{x:.4f}<br>Count: %{y}<extra></extra>',
), row=1, col=1)

# Explicit "Outlier" Indicator
fig.add_annotation(
    x=x_limit, y=max(h)*0.8,
    text=f"▶ {tail_count:,} items ({tail_pct:.1f}%)<br>continue in tail to max=1.0",
    showarrow=True, arrowhead=2, ax=-60, ay=0,
    bgcolor="rgba(192,57,43,0.8)", bordercolor="#c0392b",
    font=dict(size=11, color="white"), row=1, col=1
)

# Row 2: ECDF
sl = np.sort(lambdas)
ey = np.arange(1, len(sl)+1) / len(sl)

fig.add_trace(go.Scatter(
    x=sl, y=ey, mode='lines',
    line=dict(color='#4a90d9', width=3),
    fill='tozeroy', fillcolor='rgba(74,144,217,0.05)',
    name='Cumulative',
    hovertemplate='λ ≤ %{x:.4f}<br>Percentile: %{y:.1%}<extra></extra>'
), row=2, col=1)

# Vertical threshold lines for p25, Median, p75
for val, label, col in [
    (np.percentile(lambdas, 25), 'p25', '#4ecdc4'),
    (np.median(lambdas), 'Median', '#ffcc00'),
    (np.percentile(lambdas, 75), 'p75', '#ff6b6b'),
]:
    fig.add_vline(x=val, line_dash='dot', line_color=col, line_width=1, row=2, col=1)
    fig.add_annotation(x=val, y=0.1, text=label, font=dict(color=col, size=10), 
                       showarrow=False, textangle=-90, xshift=10, row=2, col=1)

# --- 3. Layout & Style ---
fig.update_layout(
    template="plotly_dark",
    paper_bgcolor='rgb(10, 10, 15)',
    plot_bgcolor='rgb(15, 15, 25)',
    height=800,
    title=dict(
        text=f"Spectral Fingerprint: {len(lambdas):,} Samples",
        font=dict(size=18, color="#0E87FF"), x=0.5
    ),
    margin=dict(t=120)
)

# Set the focus range
fig.update_xaxes(range=[0, x_limit], row=1, col=1, title_text="λ (Rayleigh quotient)")
fig.update_xaxes(range=[0, 1.0], row=2, col=1, title_text="λ (Full Range for ECDF)")
fig.update_yaxes(title_text="Frequency", row=1, col=1, gridcolor='rgba(255,255,255,0.05)')
fig.update_yaxes(title_text="Percentile", tickformat='.0%', row=2, col=1, gridcolor='rgba(255,255,255,0.05)')

fig.show()

ADD a distribution descriptions

## Search and documents retrival:

In [21]:
dataset[0]

{'id': 'pk_03138',
 'author_reputation': 65,
 'version': 2,
 'fork_count': 19,
 'likes': 478,
 'upvotes': 277,
 'downvotes': 6,
 'views': 2770,
 'uses': 776,
 'created_at': '2024-08-25T00:47:27Z',
 'title': 'Complete Brand Identity Brief',
 'content': 'Create a comprehensive brand identity brief document for a new sustainable fashion startup. Include all essential sections: Executive Summary, Brand Purpose and Mission, Target Audience Persona (demographics, psychographics, behaviors), Brand Positioning statement, Brand Personality (with detailed trait descriptions), Competitive Analysis summary, Core Brand Values, Brand Promise, Tone of Voice guidelines (with examples for different scenarios), and Visual Direction overview. Format this as a professional document a design agency would deliver to stakeholders.',
 'category': 'brand-identity',
 'subcategory': 'strategic-brief',
 'tags': ['brand-brief', 'sustainable-fashion', 'strategy', 'identity'],
 'has_placeholders': False,
 'placehold

In [22]:

q_embs = np.load(query_path).astype(np.float64)
idx    = json.load(open(query_index_path))

row    = idx["q_09"]["row_index"]          # 8
query  = q_embs[row].astype(np.float64)

In [23]:
ids_path = ROOT / "data" / "nomic_embs" / "embeddings_nomic_structured_768d_ids.npy"

In [24]:

# Build a lookup: id → full record
dataset_map = {r["id"]: r for r in dataset}

# ── Salience weights ──────────────────────────────────────────────────
W_UP   = 0.35
W_LK   = 0.35
W_REP  = 0.20
W_VIEW = 0.10

def _norm(arr):
    mn, mx = arr.min(), arr.max()
    return (arr - mn) / (mx - mn + 1e-9)

# Use dataset_map instead of TinyDB — TinyDB only has id + doc_string
ids        = list(dataset_map.keys())
records    = [dataset_map[i] for i in ids]

upvotes    = _norm(np.array([r.get("upvotes", 0)           for r in records], dtype=float))
likes      = _norm(np.array([r.get("likes", 0)             for r in records], dtype=float))
reputation = _norm(np.array([r.get("author_reputation", 0) for r in records], dtype=float))
views      = _norm(np.log1p(np.array([r.get("views", 0)    for r in records], dtype=float)))

salience_raw = W_UP * upvotes + W_LK * likes + W_REP * reputation + W_VIEW * views
salience_arr = _norm(salience_raw)

salience_map = {ids[i]: float(salience_arr[i]) for i in range(len(ids))}

In [25]:
# ── MMR: Maximal Marginal Relevance + Salience ───────────────────────
LAM        = 1.0
SAL_WEIGHT = 0.30
N_RETRIEVE = 10

def mmr_rerank(candidates, embs, ids, lam=LAM, k=N_RETRIEVE,
               sal_weight=SAL_WEIGHT, sal_map=None):
    """
    candidates : list of (row_index, cosine_score)  — raw aspace output
    embs       : np.ndarray  shape (N, D), indexed by row_index
    ids        : list of prompt_ids, indexed by row_index
    sal_map    : dict  prompt_id → salience in [0, 1]
    """
    if sal_map is None:
        sal_map = {}

    def relevance(i):
        ri  = candidates[i][0]          # row index into embs / ids
        cos = candidates[i][1]          # cosine score
        pid = ids[ri]                   # prompt id
        sal = sal_map.get(pid, 0.0)
        return (1.0 - sal_weight) * cos + sal_weight * sal

    selected, remaining = [], list(range(len(candidates)))

    while len(selected) < k and remaining:
        if not selected:
            best = max(remaining, key=relevance)
        else:
            sel_embs = np.array([embs[candidates[i][0]] for i in selected])

            def mmr_score(i):
                e     = embs[candidates[i][0]]
                norms = np.linalg.norm(sel_embs, axis=1) * np.linalg.norm(e) + 1e-9
                sim_to_selected = np.max(sel_embs @ e / norms)
                return lam * relevance(i) - (1 - lam) * sim_to_selected

            best = max(remaining, key=mmr_score)

        selected.append(best)
        remaining.remove(best)

    return [candidates[i] for i in selected]

In [26]:
from IPython.display import display, HTML
import numpy as np


ids        = list(np.load(ids_path))
id_to_row  = {pk: i for i, pk in enumerate(ids)}


ALPHA        = 0.6
N_RETRIEVE   = 10
SPOT_QUERIES = list(idx.keys())
pk_meta      = {item["id"]: item for item in dataset}


# ── palette ───────────────────────────────────────────────────────────────────
BG   = "#0f1117"; SURF = "#1a1d27"; SURF2 = "#22263a"
BDR  = "#2e3350"; TXT  = "#e2e6f0"; MUT   = "#7a82a0"
ASP  = "#a86fdf"; GRN  = "#6daa45"; RED   = "#dd6974"; ORG  = "#fdab43"
TEAL = "#4f98a3"; YLW  = "#e8b84b"; AMBER = "#fdab43"; BLUE = "#4a90d9"


def prompt_snippet(pk, max_chars=140):
    item = pk_meta.get(pk, {})
    body = item.get("content", "—")
    snip = body[:max_chars] + ("…" if len(body) > max_chars else "")
    return snip, item.get("title", pk), item.get("category", ""), item.get("difficulty", "")


def _rank_badge(rank, total):
    col = GRN if rank <= 2 else (RED if rank >= total - 1 else MUT)
    return (f'<span style="background:{SURF2};color:{col};font-family:monospace;'
            f'font-size:12px;font-weight:700;padding:2px 8px;border-radius:5px">#{rank}</span>')


def _target_badge(rank, total, not_found=False):
    if not_found:
        return (f'<span style="background:{SURF2};color:{RED};font-family:monospace;'
                f'font-size:11px;font-weight:700;padding:2px 8px;border-radius:5px">'
                f'✗ not in top-{total}</span>')
    col = GRN if rank <= 3 else (YLW if rank <= total // 2 else RED)
    return (f'<span style="background:{SURF2};color:{col};font-family:monospace;'
            f'font-size:11px;font-weight:700;padding:2px 8px;border-radius:5px">'
            f'🎯 #{rank}/{total}</span>')


def _result_row(rank, pk, score, expected_pk, total, highlight_color, sal_map=None):
    snip, title, cat, diff = prompt_snippet(pk)
    is_target = pk == expected_pk
    row_bg    = "#162816" if is_target else SURF
    border_l  = f"border-left:3px solid {highlight_color}" if is_target else ""
    tgt_badge = (f' <span style="color:{GRN};font-size:10px;font-weight:700;'
                 f'background:#1e3a1e;padding:1px 5px;border-radius:3px">TARGET</span>'
                 if is_target else "")
    item = pk_meta.get(pk, {})
    ups  = item.get("upvotes", 0)
    lks  = item.get("likes", 0)
    sal  = sal_map.get(pk, 0.0) if sal_map else 0.0
    eng  = (f'<span style="color:{MUT};font-size:10px">▲{ups} ♥{lks} '
            f'<span style="color:{TEAL}">✦{sal:.2f}</span></span>')
    return f"""
<tr style="background:{row_bg};border-bottom:1px solid {BDR};{border_l}">
  <td style="padding:6px 10px;vertical-align:top;white-space:nowrap">
    {_rank_badge(rank, total)}
  </td>
  <td style="padding:6px 10px;vertical-align:top;max-width:200px">
    <code style="color:{ASP};font-size:11px">{pk}</code>{tgt_badge}<br>
    <span style="color:{TXT};font-weight:600;font-size:12px">{title}</span><br>
    <span style="color:{MUT};font-size:10px">{cat}</span><br>{eng}
  </td>
  <td style="padding:6px 10px;vertical-align:top;font-family:monospace;
      color:{MUT};font-size:11px;max-width:300px;word-break:break-word">{snip}</td>
  <td style="padding:6px 10px;vertical-align:middle;text-align:right;
      font-family:monospace;color:{ASP};font-size:12px;font-weight:700;
      white-space:nowrap">{score:.5f}</td>
</tr>"""


def _separator_row(hidden):
    return (f'<tr style="background:{SURF2}"><td colspan="4" style="padding:4px 10px;'
            f'color:{MUT};font-size:10px;text-align:center">·· {hidden} hidden ··</td></tr>')


def _normalise_hits(hits):
    """Accept (ri, sc) from aspace OR (ri, sem, sal, final) from salience reranker."""
    out = []
    for h in hits:
        if len(h) == 4:
            out.append((h[0], h[3]))   # final blended score
        else:
            out.append((h[0], h[1]))   # raw cosine
    return out


def _build_table(hits, ids, expected_pk, n_retrieve, highlight_color, sal_map=None):
    hits  = _normalise_hits(hits)
    top_n = hits[:n_retrieve]
    total = len(top_n)

    target_rank = next(
        (r for r, (ri, _) in enumerate(top_n, 1) if ids[ri] == expected_pk), None
    )

    top3          = top_n[:3]
    bot_start_idx = max(3, total - 3)
    bottom3       = top_n[bot_start_idx:]
    hidden        = max(0, total - 6)

    rows_html = ""
    for rank, (ri, sc) in enumerate(top3, 1):
        rows_html += _result_row(rank, ids[ri], sc, expected_pk, total,
                                 highlight_color, sal_map)
    if hidden > 0:
        rows_html += _separator_row(hidden)
    for rank, (ri, sc) in enumerate(bottom3, bot_start_idx + 1):
        rows_html += _result_row(rank, ids[ri], sc, expected_pk, total,
                                 highlight_color, sal_map)

    badge = _target_badge(target_rank, total, not_found=(target_rank is None))
    return rows_html, badge, total, target_rank


def _col_header(label, color, badge, total):
    return f"""
<div style="background:{SURF2};padding:8px 12px;border-bottom:2px solid {color}">
  <span style="color:{color};font-weight:700;font-size:13px">{label}</span>
  <span style="float:right">{badge}
    <span style="color:{MUT};font-size:11px;margin-left:8px">{total} results</span>
  </span>
</div>
<table style="width:100%;border-collapse:collapse;background:{SURF}">
  <tr style="background:{SURF2}">
    <th style="padding:5px 10px;color:{MUT};font-size:10px;text-align:left;width:50px">Rank</th>
    <th style="padding:5px 10px;color:{MUT};font-size:10px;text-align:left;width:200px">Prompt</th>
    <th style="padding:5px 10px;color:{MUT};font-size:10px;text-align:left">Preview</th>
    <th style="padding:5px 10px;color:{MUT};font-size:10px;text-align:right;width:80px">Score</th>
  </tr>"""


# ── main render ───────────────────────────────────────────────────────────────
html = (f'<div style="background:{BG};padding:20px;border-radius:14px;'
        f'font-family:system-ui,sans-serif;max-width:1600px">'
        f'<h2 style="color:{TXT};margin:0 0 4px">ArrowSpace Spot-Check — Base vs MMR+Salience</h2>'
        f'<p style="color:{MUT};font-size:12px;margin:0 0 18px">'
        f'α={ALPHA} · top-3 and bottom-3 of {N_RETRIEVE} · '
        f'<span style="color:{BLUE}">■ Base</span>  '
        f'<span style="color:{AMBER}">■ MMR+Salience (λ={LAM}, sal={SAL_WEIGHT})</span>'
        f'  <span style="color:{TEAL}">✦ = salience score</span></p>')

wins_mmr = wins_base = ties = 0

for qid in SPOT_QUERIES:
    meta_q      = idx[qid]
    row_i       = meta_q["row_index"]
    expected_pk = meta_q["expected_prompt_id"]
    query_type  = meta_q.get("query_type", "")
    query_text  = meta_q.get("query_text", meta_q.get("query", ""))
    q_vec       = q_embs[row_i].astype(np.float64)

    # ── Base: raw aspace search ───────────────────────────────────────────────

    hits_base  = aspace.search(q_vec, gl, ALPHA)
    candidates = aspace.search(q_vec, gl, ALPHA)
    hits_mmr   = mmr_rerank(candidates, embs, ids, lam=LAM, k=N_RETRIEVE,
                             sal_weight=SAL_WEIGHT, sal_map=salience_map)

    rows_base, badge_base, total_base, rank_base = _build_table(
        hits_base, ids, expected_pk, N_RETRIEVE, BLUE)
    rows_mmr,  badge_mmr,  total_mmr,  rank_mmr  = _build_table(
        hits_mmr,  ids, expected_pk, N_RETRIEVE, AMBER, sal_map=salience_map)

    # ── win indicator ─────────────────────────────────────────────────────────
    r_b = rank_base if rank_base else 999
    r_m = rank_mmr  if rank_mmr  else 999
    if r_m < r_b:
        win_label = f'<span style="color:{GRN};font-weight:700">▲ MMR wins (#{r_m} vs #{r_b})</span>'
        wins_mmr += 1
    elif r_b < r_m:
        win_label = f'<span style="color:{BLUE};font-weight:700">▲ BASE wins (#{r_b} vs #{r_m})</span>'
        wins_base += 1
    else:
        win_label = f'<span style="color:{MUT}">= TIE (both #{r_b})</span>'
        ties += 1

    query_block = ""
    if query_text:
        query_block = (f'<div style="background:{BG};border-left:3px solid {TEAL};'
                       f'padding:8px 12px;margin:8px 0 0;font-size:12px;color:{TXT};'
                       f'font-style:italic">'
                       f'<span style="color:{TEAL};font-style:normal;font-weight:600;'
                       f'font-size:10px;display:block;margin-bottom:3px">QUERY</span>'
                       f'{query_text}</div>')

    html += f"""
<div style="border:1px solid {BDR};border-radius:10px;overflow:hidden;margin-bottom:20px">
  <div style="background:{SURF2};padding:10px 14px;display:flex;
      align-items:center;gap:10px;flex-wrap:wrap">
    <span style="font-size:15px;font-weight:700;color:{TXT}">{qid}</span>
    <span style="background:{BG};color:{MUT};font-size:11px;padding:2px 8px;
        border-radius:999px">{query_type}</span>
    <span style="font-size:11px;color:{MUT}">target:
      <code style="color:{ORG};background:{BG};padding:1px 6px;
          border-radius:3px">{expected_pk}</code>
    </span>
    <span style="margin-left:auto;font-size:12px">{win_label}</span>
  </div>
  {query_block}
  <div style="display:grid;grid-template-columns:1fr 1fr;gap:0;
      border-top:1px solid {BDR}">
    <div style="border-right:1px solid {BDR}">
      {_col_header("Base — semantic only", BLUE, badge_base, total_base)}
      {rows_base}
      </table>
    </div>
    <div>
      {_col_header("MMR+Salience reranked", AMBER, badge_mmr, total_mmr)}
      {rows_mmr}
      </table>
    </div>
  </div>
</div>"""

# ── summary scoreboard ────────────────────────────────────────────────────────
total_q = len(SPOT_QUERIES)
html += f"""
<div style="background:{SURF2};border-radius:10px;padding:16px 20px;
    display:flex;gap:32px;align-items:center;flex-wrap:wrap">
  <span style="color:{TXT};font-weight:700;font-size:14px">Scoreboard</span>
  <span style="color:{AMBER};font-weight:700">MMR+Sal wins: {wins_mmr}/{total_q}</span>
  <span style="color:{BLUE};font-weight:700">Base wins: {wins_base}/{total_q}</span>
  <span style="color:{MUT}">Ties: {ties}/{total_q}</span>
</div>"""

html += '</div>'
display(HTML(html))